<a href="https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule in plain words (written before coding it):** *A page is worth reviewing first if it still gets real search demand (at least 500 impressions in 90 days), sits on page one or two (average position 20 or better), and its click-through rate is far below what pages at the same position normally get. Its title and snippet are probably not doing their job.*

**Lane check:** I am staying with **Lane 2 (Refresh / Content Opportunity Scoring)**; the lock for this week is a confirm, not a switch. (Edit this line if you decide differently.)

**Data:** the 30,000-row starter CSV in my repo (one row = one page, trailing 90 days). The "declining" label (`trend_direction == "down"`) is only used to *check* signals and to *score* the baseline. It is never an input to the rule.

Before I encode anything, I check two signals the rule leans on. One is behind a real FlyRank flag (CTR vs. position, the CTR-fix logic), and the other is staleness (behind the refresh flags).

In [1]:
import os, sys, subprocess, json
import numpy as np
import pandas as pd

# Find the repo root so the CSV loads in Colab and locally.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yelsebety/Yassin-Flyrank.git"
REPO_DIR = "Yassin-Flyrank"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The proxy label: used ONLY to check signals and score the baseline, never as an input.
raw["declined"] = (raw["trend_direction"] == "down").astype(int)

# The slice the rule works on: starter eligibility + the rule's gate.
elig = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)]
g = elig[(elig["impressions_90d"] >= 500) & (elig["avg_position"] > 0) & (elig["avg_position"] <= 20)].copy()
g["band"] = pd.cut(g["avg_position"], [0, 3, 10, 20], labels=["pos 1-3", "pos 4-10", "pos 11-20"])

print(f"Eligible pages: {len(elig):,}. Pages passing the rule's gate: {len(g):,} "
      f"(500+ impressions, position 1-20).")
print(f"Base rate of the proxy label in the gated slice: {g['declined'].mean():.3f}")

Eligible pages: 30,000. Pages passing the rule's gate: 12,023 (500+ impressions, position 1-20).
Base rate of the proxy label in the gated slice: 0.599


### Signal 1 (behind a real flag): CTR vs. position

**Idea behind the CTR-fix logic:** pages at the same position should earn a similar CTR. If a page earns much less, its title/snippet may be the problem, and such a page should be more likely to be losing visibility. **My expectation before looking:** within a position band, the lowest-CTR quartile should decline more often than the highest-CTR quartile.

(CTR is a ×100 percentage here. Quartiles are cut *inside* each position band, so position is not mixed into CTR.)

In [2]:
g["ctr_quartile"] = g.groupby("band", observed=True)["ctr"].transform(
    lambda s: pd.qcut(s.rank(method="first"), 4, labels=["Q1 lowest CTR", "Q2", "Q3", "Q4 highest CTR"])
)
signal1 = (g.groupby(["band", "ctr_quartile"], observed=True)["declined"]
             .agg(n="count", decline_rate="mean").round(3))
signal1

n  decline_rate
band      ctr_quartile                      
pos 1-3   Q1 lowest CTR    120         0.875
          Q2               120         0.883
          Q3               120         0.808
          Q4 highest CTR   120         0.383
pos 4-10  Q1 lowest CTR   1771         0.702
          Q2              1771         0.590
          Q3              1771         0.551
          Q4 highest CTR  1771         0.472
pos 11-20 Q1 lowest CTR   1115         0.705
          Q2              1115         0.612
          Q3              1114         0.588
          Q4 highest CTR  1115         0.555

**Verdict 1: CONFIRMED.** In position 4–10 (n = 1,771 per quartile) the decline rate falls steadily from 0.702 for the lowest-CTR quartile to 0.472 for the highest. Position 11–20 (n ≈ 1,115 per quartile) shows the same direction, 0.705 down to 0.555. In position 1–3 the bottom three quartiles are all high (0.81 to 0.88) and only the top quartile drops (0.383), but n is just 120 per bucket there, so I treat it as supporting evidence, not proof. Overall the direction matches my expectation, so CTR-below-position is a usable signal for the rule. This is an association with a proxy label, not proof that a better title would stop a decline.

### Signal 2: staleness (behind the refresh flags)

**Idea behind the refresh flags:** the longer a page goes without an update, the more likely it is losing visibility. **My expectation before looking:** decline rate rises with `days_since_last_update`.

In [3]:
g["staleness"] = pd.cut(g["days_since_last_update"], [0, 30, 90, 180, 10**6], right=False,
                        labels=["0-29 d", "30-89 d", "90-179 d", "180+ d"])
signal2 = (g.groupby("staleness", observed=True)["declined"]
             .agg(n="count", decline_rate="mean").round(3))
print(f"Share of gated pages updated within the last 30 days: {(g['days_since_last_update'] < 30).mean():.1%}")
signal2

Share of gated pages updated within the last 30 days: 63.3%


,n,decline_rate
staleness,,
0-29 d,7613,0.587
30-89 d,60,0.450
90-179 d,4340,0.620
180+ d,10,1.000


**Verdict 2: MIXED.** The 90–179 day band is a little higher than the freshest band (0.620 vs. 0.587), which points the expected way. But the 30–89 day band is *lower* (0.450, n = 60), and the 180+ day band has only n = 10 pages (all declined, which is far too few to trust). So the relationship is not clean, and most gated pages (about 63%) were updated within the last 30 days, so there is little real staleness to learn from in this slice. **What this means for my rule:** staleness gets no weight in the score (it would be noise). I still show it in the top-10 review as context. The check saved me from a rule built on a signal this data can't support.

### The rule, built from those verdicts

- **Gate:** at least 500 impressions in 90 days and average position 1–20 (volume is a *gate*, not a weight, because I did not confirm that volume itself predicts decline).
- **Score:** `ctr_shortfall = (typical CTR for the position band − page CTR) / typical CTR`, floored at 0, where the typical CTR is the median CTR of gated pages in the same band. Score runs from 0 (at or above typical) to 1 (zero CTR). Ties are broken by impressions (more demand first).
- **One reason code:** `low_ctr_for_position`.
- **Action label:** `fix_title_and_snippet` (a reviewer rewrites the title and meta description and checks the snippet).

**An honest note on how I chose the score.** I first tried a value-style score, `impressions × CTR gap` ("missed clicks"). On the proxy label it did *worse than the base rate*. I then used the shortfall score above, which follows directly from the verdict on signal 1. Because I saw both results before settling, the number I report for the second one is a little optimistic, and I say so in the metrics file.

## 2. Build the ranked queue (writes the CSV)

Code the score, rank everything, write `work/outputs/baseline_action_score.csv` (this CSV stays out of git by design) and a small metrics JSON (the run's receipt, which is safe to commit).

In [4]:
REASON_CODE = "low_ctr_for_position"
ACTION      = "fix_title_and_snippet"

# Typical CTR per position band = median CTR of gated pages in that band (no label used).
typical = g.groupby("band", observed=True)["ctr"].median()
g["typical_ctr"] = g["band"].map(typical).astype(float)

# Transparent score: relative CTR shortfall, floored at 0. No fitted weights.
g["score"] = ((g["typical_ctr"] - g["ctr"]) / g["typical_ctr"]).clip(lower=0)
g["reason_code"] = np.where(g["score"] > 0, REASON_CODE, "no_flag")
g["action"] = np.where(g["score"] > 0, ACTION, "monitor")

queue = g.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

out_cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
            "impressions_90d", "avg_position", "ctr", "typical_ctr"]      # label is NOT written
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv  ({len(queue):,} rows)")
print("Typical CTR per band (median, %):", typical.round(3).to_dict())

# precision@K on the proxy label, always printed next to the base rate
def precision_at_k(ranked_labels, k):
    return float(np.asarray(ranked_labels)[:k].mean())

base_rate = float(g["declined"].mean())
naive = g.sort_values("impressions_90d", ascending=False)["declined"].to_numpy()
missed_clicks = (g["impressions_90d"] * (g["typical_ctr"] - g["ctr"]).clip(lower=0) / 100)
variant_a = g.assign(_s=missed_clicks).sort_values("_s", ascending=False)["declined"].to_numpy()

rows = []
for k in (20, 50):
    rows.append({"K": k,
                 "my rule (shortfall)": precision_at_k(queue["declined"], k),
                 "variant tried first (missed clicks)": precision_at_k(variant_a, k),
                 "naive (most impressions first)": precision_at_k(naive, k),
                 "base rate": base_rate})
results = pd.DataFrame(rows).round(3)
display(results)

metrics = {
    "slice": "starter CSV, gate: impressions_90d>=500, 0<avg_position<=20, content_age_days>=90",
    "n_scored": int(len(queue)),
    "rule": "ctr_shortfall = max(0, (typical_ctr_for_band - ctr) / typical_ctr_for_band); ties broken by impressions_90d",
    "reason_code": REASON_CODE, "action_label": ACTION,
    "inputs_used": ["impressions_90d", "avg_position", "ctr", "content_age_days"],
    "label_used_only_for_evaluation": "trend_direction == 'down' (proxy)",
    "base_rate": round(base_rate, 3),
    "precision_at_20": round(precision_at_k(queue["declined"], 20), 3),
    "precision_at_50": round(precision_at_k(queue["declined"], 50), 3),
    "naive_precision_at_50": round(precision_at_k(naive, 50), 3),
    "variant_tried_first_precision_at_50": round(precision_at_k(variant_a, 50), 3),
    "caveat": "Two score variants were compared before settling, so the reported precision is slightly optimistic. "
              "The proxy label comes from the same 90-day window as ctr/impressions, so this is a method check, not a forecast.",
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w04_baseline_metrics.json")

Wrote work/outputs/baseline_action_score.csv  (12,023 rows)
Typical CTR per band (median, %): {'pos 1-3': 0.2, 'pos 4-10': 0.24, 'pos 11-20': 0.17}


,K,my rule (shortfall),variant tried first (missed clicks),naive (most impressions first),base rate
0,20,0.70,0.50,0.45,0.599
1,50,0.78,0.54,0.42,0.599


Wrote work/outputs/w04_baseline_metrics.json


## 3. Top-10 review

For each of the top ten: the action, why it is there, and what would make it wrong. The "why" lines come straight from each page's numbers. The "what would make it wrong" lines name the specific doubt for that page (a data doubt, a recent-update doubt, and an intent doubt).

In [5]:
top10 = queue.head(10).copy()

def why(r):
    return (f"{r.impressions_90d:,.0f} impressions at position {r.avg_position:.1f}, but CTR {r.ctr:.2f}% "
            f"vs. {r.typical_ctr:.2f}% typical for {r.band} (shortfall {r.score:.0%}).")

def wrong_if(r):
    bits = []
    if r.clicks_90d == 0:
        bits.append(f"0 clicks from {r.impressions_90d:,.0f} impressions at position {r.avg_position:.1f} is hard to believe, "
                    "so this may be a tracking or measurement gap, not a title problem")
    if r.days_since_last_update < 30:
        bits.append(f"it was updated only {r.days_since_last_update:.0f} days ago, so CTR may not have caught up yet")
    if r.main_intent == "informational":
        bits.append("the query may be answered in the search snippet itself (a zero-click search), so a low CTR is normal")
    elif r.main_intent == "transactional":
        bits.append("ads or shopping boxes above the page may be taking the clicks, so the title is not the problem")
    elif r.main_intent == "commercial":
        bits.append("comparison boxes in the results may be taking the clicks, so the title is not the problem")
    else:
        bits.append("the search-intent field is blank, so I cannot tell what a normal CTR is for it")
    return "Wrong if " + "; or ".join(bits) + "."

top10["action_label"] = top10["action"]
top10["why_it_is_here"] = top10.apply(why, axis=1)
top10["what_would_make_it_wrong"] = top10.apply(wrong_if, axis=1)
review = top10[["rank", "content_id", "action_label", "why_it_is_here", "what_would_make_it_wrong"]]
for _, r in review.iterrows():
    print(f"#{r['rank']}  {r['content_id']}  ->  {r['action_label']}")
    print(f"    why:   {r['why_it_is_here']}")
    print(f"    wrong: {r['what_would_make_it_wrong']}")

#1  content_c8e9d6ab9013  ->  fix_title_and_snippet
    why:   208,678 impressions at position 9.7, but CTR 0.00% vs. 0.24% typical for pos 4-10 (shortfall 100%).
    wrong: Wrong if 0 clicks from 208,678 impressions at position 9.7 is hard to believe, so this may be a tracking or measurement gap, not a title problem; or the query may be answered in the search snippet itself (a zero-click search), so a low CTR is normal.
#2  content_f986bd514b6e  ->  fix_title_and_snippet
    why:   22,456 impressions at position 6.6, but CTR 0.00% vs. 0.24% typical for pos 4-10 (shortfall 100%).
    wrong: Wrong if it was updated only 20 days ago, so CTR may not have caught up yet; or the search-intent field is blank, so I cannot tell what a normal CTR is for it.
#3  content_ae6d1339904d  ->  fix_title_and_snippet
    why:   17,622 impressions at position 19.5, but CTR 0.00% vs. 0.17% typical for pos 11-20 (shortfall 100%).
    wrong: Wrong if 0 clicks from 17,622 impressions at position 19.5 is hard 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# After-the-fact check using the proxy label (a review aid, never an input)
weak = top10[top10["declined"] == 0][["rank", "content_id", "impressions_90d", "avg_position", "ctr", "trend_direction"]]
print(f"Of my top 10, {int(top10['declined'].sum())} are 'down' on the proxy label and {len(weak)} are not (weak picks):")
display(weak)

zero_click_share = (g["clicks_90d"] == 0).mean()
print(f"\nGated pages with 0 clicks in 90 days: {int((g['clicks_90d'] == 0).sum()):,} ({zero_click_share:.1%}); "
      f"of the top 10, {int((top10['clicks_90d'] == 0).sum())} have 0 clicks.")

# Leakage guard: nothing label-derived or future-window may be an input; IDs are never inputs.
INPUTS = ["impressions_90d", "avg_position", "ctr", "content_age_days"]          # what the score reads (+ band from avg_position)
BANNED = ["trend_direction", "trend_pct", "declined", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"]
assert not set(INPUTS) & set(BANNED), "a banned column is an input"
written = pd.read_csv("work/outputs/baseline_action_score.csv", nrows=1).columns
assert not {"declined", "trend_direction", "trend_pct"} & set(written), "the label is in the output CSV"
print("\nLeakage check passed: inputs =", INPUTS)
print("No label, trend or last/prev-30-day columns are inputs; the CSV contains no label.")

Of my top 10, 6 are 'down' on the proxy label and 4 are not (weak picks):


,rank,content_id,impressions_90d,avg_position,ctr,trend_direction
5,6,content_5d5653c4eb4f,15101,5.7,0.0,stable
6,7,content_847a841969a2,14519,7.4,0.0,stable
8,9,content_eb1510f4b5f1,12275,14.7,0.0,stable
9,10,content_3e79eaafc89d,8779,11.6,0.0,stable



Gated pages with 0 clicks in 90 days: 1,215 (10.1%); of the top 10, 9 have 0 clicks.

Leakage check passed: inputs = ['impressions_90d', 'avg_position', 'ctr', 'content_age_days']
No label, trend or last/prev-30-day columns are inputs; the CSV contains no label.


**Weak picks, in words.** Four of my top ten (ranks 6, 7, 9 and 10) are "stable" on the proxy label, so they are weak picks by that measure. The bigger worry is something else: **9 of my top 10 have exactly 0 clicks in 90 days** despite thousands of impressions at position 4–20. That is a pattern I do not believe by default. It may be a real title problem, but it may also be a tracking or measurement gap that shows up as CTR = 0. So the top of this list could be catching a *data* problem as much as a *content* problem.

**What that means for Week 5:** the baseline still beats the base rate on the proxy label (precision@50 of 0.78 vs. 0.599), but I should check how the ranking behaves *beyond* the CTR = 0 pages before I trust it. For example, I can score only pages with at least some clicks and see whether it holds up. A model has to beat this baseline in a fairer test than that.

**Leakage check, in words.** The score reads only `impressions_90d`, `avg_position`, `ctr` and the `content_age_days` gate. The label (`trend_direction`), the trend columns and the last/prev-30-day columns are never inputs, and the output CSV does not contain the label. One caveat I cannot remove: the proxy label and the 90-day CTR come from the same data window, so this checks the method; it is not a forecast of future declines.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.